# 🤖 Shopee AI Co-Pilot — Model Server
Colab-only inference server untuk agent dashboard.

**Cara pakai:**
1. 🔑 Set Colab Secret: klik ikon kunci di sidebar kiri → Add secret `NGROK_AUTH` → paste token ngrok → toggle ON
2. Jalankan semua cell dari atas ke bawah
3. Copy URL ngrok yang muncul
4. Masukkan URL tersebut ke dashboard settings

**Catatan:**
- Model: Qwen 2.5 7B Instruct (4-bit)
- VRAM: ~6-7GB dari 15GB T4
- Session Colab ~4-8 jam (free tier)
- Token ngrok cukup diset SEKALI via Colab Secrets, otomatis kepakai tiap sesi

In [ ]:
# @title 📦 Install Dependencies
import subprocess, sys, importlib, pkg_resources

reqs = [
    "torch>=2.1.0",
    "transformers>=4.38.0",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
    "fastapi>=0.109.0",
    "uvicorn>=0.27.0",
    "pyngrok>=7.0.0",
    "pydantic>=2.5.0",
    "python-multipart>=0.0.6",
    "pandas>=2.1.0",
    "openpyxl>=3.1.0",
    "nest-asyncio>=1.5.8",
]

for r in reqs:
    pkg_name = r.split(">=")[0].split("==")[0]
    try:
        importlib.import_module(pkg_name.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", r])
        print(f"Installed: {r}")

print("\n✅ Semua dependencies siap")

In [ ]:
# @title 🔗 Mount Google Drive & Setup Path
from google.colab import drive
import sys, os

drive.mount('/content/drive')

# Buat folder utils di Drive jika belum ada
utils_path = '/content/drive/MyDrive/shopee_agent/utils'
os.makedirs(utils_path, exist_ok=True)

# Upload utils dari project kita ke Drive (jalankan sekali)
# atau langsung download dari repo
if not os.path.exists(f'{utils_path}/agent_orchestrator.py'):
    print("⚠️  Folder utils kosong. Upload file dari repo ke:")
    print(f"   {utils_path}")
    print("   Atau jalankan cell berikutnya untuk download otomatis.")

sys.path.insert(0, utils_path)
print(f"✅ Google Drive ter-mount. Utils path: {utils_path}")

In [ ]:
# @title ⬇️ Download Utils dari GitHub (alternatif)
# Kalau file utils belum di-upload, jalankan ini sekali

import urllib.request

UTILS_REPO = "https://raw.githubusercontent.com/your-org/shopee-ai-co-pilot/main/colab/utils"
FILES = [
    "agent_orchestrator.py",
    "title_doctor.py",
    "keyword_genius.py",
    "desc_master.py",
    "pricing_intel.py",
    "seasonal_pro.py",
    "review_intel.py",
]

for f in FILES:
    url = f"{UTILS_REPO}/{f}"
    dest = f"{utils_path}/{f}"
    if not os.path.exists(dest):
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"Downloaded: {f}")
        except Exception as e:
            print(f"Gagal download {f}: {e}")

print("\n✅ Selesai. Reload module.")
import importlib
importlib.invalidate_caches()

In [ ]:
# @title 📥 Import Libraries
import torch
import json
import asyncio
import re
import pandas as pd
from typing import Optional, List, Dict, Any
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# @title 🧠 Load Model: Qwen 2.5 7B (4-bit)
# @markdown Butuh ~5 menit untuk pertama kali (download model)
# @markdown Selanjutnya ~30 detik (load dari cache)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)

model.eval()
print(f"\n✅ Model loaded! VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# @title 🔧 Load Subagent Utils
try:
    from agent_orchestrator import AgentOrchestrator
    from title_doctor import TitleDoctor
    from keyword_genius import KeywordGenius
    from desc_master import DescMaster
    from pricing_intel import PricingIntel
    from seasonal_pro import SeasonalPro
    from review_intel import ReviewIntel

    # Init orchestrator
    orchestrator = AgentOrchestrator(model, tokenizer)
    orchestrator.register_subagent('title_optimization', TitleDoctor(model, tokenizer))
    orchestrator.register_subagent('keyword_research', KeywordGenius(model, tokenizer))
    orchestrator.register_subagent('description_writing', DescMaster(model, tokenizer))
    orchestrator.register_subagent('pricing_analysis', PricingIntel(model, tokenizer))
    orchestrator.register_subagent('seasonal_promo', SeasonalPro(model, tokenizer))
    orchestrator.register_subagent('review_analysis', ReviewIntel(model, tokenizer))

    print("✅ Subagent utils loaded and registered")
except ImportError as e:
    print(f"⚠️  Utils belum ada di path. Error: {e}")
    print("   Upload file .py ke Google Drive atau download dari repo.")
    orchestrator = None

In [ ]:
# @title 🌐 Start FastAPI Server
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="Shopee AI Co-Pilot", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ──── Request/Response Models ────

class ChatRequest(BaseModel):
    message: str
    context: Optional[Dict[str, Any]] = None

class ChatResponse(BaseModel):
    response: str
    intent: str
    agent_trace: List[Dict[str, Any]] = []

class ProductAnalysisRequest(BaseModel):
    products: List[Dict[str, Any]]
    analysis_type: str = "all"  # all, title, keyword, desc, pricing, promo, review

class ProductAnalysisResponse(BaseModel):
    results: List[Dict[str, Any]]
    summary: str

# ──── Helper ────

async def generate(prompt: str, system: str = "") -> str:
    """Generate response dari model."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# ──── Endpoints ────

@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_NAME, "vram_gb": f"{torch.cuda.memory_allocated() / 1e9:.1f}"}


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    if orchestrator:
        result = await orchestrator.route(req.message, req.context)
        return ChatResponse(**result)

    # Fallback: direct LLM
    resp = await generate(req.message, "Kamu adalah asisten e-commerce Shopee yang ahli dalam optimasi produk bunga artificial dan lampu dekorasi.")
    return ChatResponse(response=resp, intent="general")


@app.post("/analyze_products", response_model=ProductAnalysisResponse)
async def analyze_products(req: ProductAnalysisRequest):
    if not orchestrator:
        raise HTTPException(503, "Agent orchestrator belum siap")

    results = []
    for product in req.products:
        # Run relevant subagents based on analysis_type
        product_result = {"sku": product.get("sku", ""), "recommendations": []}

        if req.analysis_type in ["all", "title"] and "title_optimization" in orchestrator.subagents:
            title_res = await orchestrator.subagents["title_optimization"].process(
                json.dumps(product), {}
            )
            product_result["title"] = title_res

        if req.analysis_type in ["all", "keyword"] and "keyword_research" in orchestrator.subagents:
            kw_res = await orchestrator.subagents["keyword_research"].process(
                json.dumps(product), {}
            )
            product_result["keywords"] = kw_res

        if req.analysis_type in ["all", "desc"] and "description_writing" in orchestrator.subagents:
            desc_res = await orchestrator.subagents["description_writing"].process(
                json.dumps(product), {}
            )
            product_result["description"] = desc_res

        if req.analysis_type in ["all", "pricing"] and "pricing_analysis" in orchestrator.subagents:
            price_res = await orchestrator.subagents["pricing_analysis"].process(
                json.dumps(product), {}
            )
            product_result["pricing"] = price_res

        results.append(product_result)

    summary = await generate(
        f"Buat ringkasan 2-3 kalimat dari hasil analisis {len(results)} produk ini: {json.dumps(results)}",
        "Kamu adalah asisten e-commerce."
    )

    return ProductAnalysisResponse(results=results, summary=summary)


@app.post("/generate_keywords")
async def generate_keywords(product_name: str, category: str = ""):
    prompt = f"""Generate 20 keyword Shopee untuk produk ini:
Produk: {product_name}
Kategori: {category}

Format: JSON array of objek {{keyword, volume_estimate, competition_level, intent}}
"""
    resp = await generate(prompt, "Kamu adalah SEO specialist Shopee. Output JSON saja.")
    # Clean response to extract JSON
    try:
        resp_clean = re.sub(r'```json|```', '', resp).strip()
        keywords = json.loads(resp_clean)
        return {"keywords": keywords}
    except:
        return {"keywords": [], "raw": resp}


@app.post("/upload_csv")
async def upload_csv(file: UploadFile = File(...)):
    import tempfile
    content = await file.read()
    path = f"/tmp/{file.filename}"
    with open(path, "wb") as f:
        f.write(content)

    if file.filename.endswith('.csv'):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)

    return {
        "filename": file.filename,
        "rows": len(df),
        "columns": df.columns.tolist(),
        "preview": json.loads(df.head(5).to_json(orient="records")),
        "products": json.loads(df.to_json(orient="records")),
    }

print("✅ FastAPI app siap")

In [ ]:
# @title 🔌 Setup ngrok Tunnel (Colab Secret)
from pyngrok import ngrok
from google.colab import userdata

# Baca token dari Colab Secrets
# Cara setting:
#   1. Klik ikon 🔑 (key) di sidebar kiri Colab
#   2. Add new secret → NGROK_AUTH → paste token-mu
#   3. Toggle "Notebook access" ON
try:
    NGROK_AUTHTOKEN = userdata.get('NGROK_AUTH')
    print(f"✅ Token terbaca dari Colab Secrets")
except Exception as e:
    NGROK_AUTHTOKEN = ""
    print(f"⚠️  Colab Secret tidak ditemukan: {e}")
    print("   Pakai fallback manual...")
    NGROK_AUTHTOKEN = ""  # @param {type:"string"}

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(8080, "http")
    PUBLIC_URL = tunnel.public_url
    print(f"\n🌐 PUBLIC URL: {PUBLIC_URL}")
    print("   Masukkan URL ini di Dashboard Settings!")
else:
    PUBLIC_URL = "http://localhost:8080"
    print("⚠️  No ngrok token. Server hanya di localhost.")

In [ ]:
# @title ▶️ Jalankan Server
import uvicorn

PORT = 8080
print(f"Starting server on port {PORT}...")
print(f"Health check: http://localhost:{PORT}/health")
print(f"Chat endpoint: http://localhost:{PORT}/chat")
print(f"Analyze endpoint: http://localhost:{PORT}/analyze_products")
print("\n⏳ Press STOP (⏹) untuk menghentikan server")
print("   Jangan tutup tab ini selama dashboard terhubung!\n")

uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

In [ ]:
# @title 🔄 Keep Alive (jalankan di cell terpisah)
# Jalankan cell ini untuk mencegah Colab disconnect
# Klik kanan → "Run cell"

import time
import requests

def keep_alive():
    url = PUBLIC_URL or f"http://localhost:{PORT}"
    while True:
        try:
            r = requests.get(f"{url}/health", timeout=10)
            print(f"[{time.strftime('%H:%M:%S')}] Server OK | VRAM: {json.loads(r.text).get('vram_gb', '?')} GB")
        except:
            print(f"[{time.strftime('%H:%M:%S')}] Server unreachable...")
        time.sleep(120)  # Ping every 2 minutes

# Uncomment to run:
# import threading
# t = threading.Thread(target=keep_alive, daemon=True)
# t.start()
print("✅ Keep-alive siap. Uncomment baris di atas untuk menjalankan.")